In [1]:
from functools import lru_cache


def count_warping_paths(k: int, m: int) -> tuple[int, int]:
    """
    Return (total_paths, no_sharp_corner_paths) for warping paths from (1,1) to (k,m).

    Allowed steps are:
      - horizontal: (1, 0)
      - vertical:   (0, 1)
      - diagonal:   (1, 1)

    A path has a sharp corner if it contains either
      (i,j), (i+1,j), (i+1,j+1)   or
      (i,j), (i,j+1), (i+1,j+1),
    which is equivalent to a local HV or VH turn.
    """
    if k < 1 or m < 1:
        raise ValueError("k and m must be positive integers")

    dk, dm = k - 1, m - 1

    # Total number of warping paths (Delannoy-style recurrence).
    total = [[0] * (dm + 1) for _ in range(dk + 1)]
    total[0][0] = 1

    for i in range(dk + 1):
        for j in range(dm + 1):
            if i == 0 and j == 0:
                continue
            t = 0
            if i > 0:
                t += total[i - 1][j]
            if j > 0:
                t += total[i][j - 1]
            if i > 0 and j > 0:
                t += total[i - 1][j - 1]
            total[i][j] = t

    # Count paths without sharp corners using state = last step type.
    # H: last step horizontal, V: vertical, D: diagonal, S: start.
    H = [[0] * (dm + 1) for _ in range(dk + 1)]
    V = [[0] * (dm + 1) for _ in range(dk + 1)]
    D = [[0] * (dm + 1) for _ in range(dk + 1)]
    S = [[0] * (dm + 1) for _ in range(dk + 1)]
    S[0][0] = 1

    for i in range(dk + 1):
        for j in range(dm + 1):
            if i == 0 and j == 0:
                continue

            # Enter (i,j) with a horizontal step from (i-1,j):
            # allowed previous last step is H, D, or S (but not V).
            if i > 0:
                H[i][j] += H[i - 1][j] + D[i - 1][j] + S[i - 1][j]

            # Enter (i,j) with a vertical step from (i,j-1):
            # allowed previous last step is V, D, or S (but not H).
            if j > 0:
                V[i][j] += V[i][j - 1] + D[i][j - 1] + S[i][j - 1]

            # Enter (i,j) with a diagonal step from (i-1,j-1):
            # always allowed.
            if i > 0 and j > 0:
                D[i][j] += (
                    H[i - 1][j - 1]
                    + V[i - 1][j - 1]
                    + D[i - 1][j - 1]
                    + S[i - 1][j - 1]
                )

    no_sharp = H[dk][dm] + V[dk][dm] + D[dk][dm] + S[dk][dm]

    return total[dk][dm], no_sharp


# Example
# total, no_sharp = count_warping_paths(4, 5)
# print(f"Total: {total}, without sharp corners: {no_sharp}")

In [6]:
for i in range(1, 11):
    total, no_sharp = count_warping_paths(i, i)
    print(f"From (1,1) to ({i},{i}): Total: {total}, without sharp corners: {no_sharp}")

From (1,1) to (1,1): Total: 1, without sharp corners: 1
From (1,1) to (2,2): Total: 3, without sharp corners: 1
From (1,1) to (3,3): Total: 13, without sharp corners: 3
From (1,1) to (4,4): Total: 63, without sharp corners: 9
From (1,1) to (5,5): Total: 321, without sharp corners: 27
From (1,1) to (6,6): Total: 1683, without sharp corners: 83
From (1,1) to (7,7): Total: 8989, without sharp corners: 259
From (1,1) to (8,8): Total: 48639, without sharp corners: 817
From (1,1) to (9,9): Total: 265729, without sharp corners: 2599
From (1,1) to (10,10): Total: 1462563, without sharp corners: 8323
